In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from datetime import timedelta

# Set visualization style
plt.style.use('ggplot')
sns.set_context("notebook", font_scale=1.2)

# Load the datasets
try:
    filter_flow_df = pd.read_parquet('filterflow_data.parquet')
    error_df = pd.read_parquet('error_data.parquet')
    
    # Convert timestamps to datetime if they aren't already
    filter_flow_df['timestamp'] = pd.to_datetime(filter_flow_df['timestamp'])
    error_df['timestamp'] = pd.to_datetime(error_df['timestamp'])
    
    # Get unique printers
    printers = sorted(filter_flow_df['printer'].unique())
    
except Exception as e:
    print(f"Error loading data: {e}")
    print("This notebook assumes the data files are in the current directory.")

In [ ]:
def visualize_printer_data(printer_id):
    """Create comprehensive visualizations for a specific printer"""
    
    # Filter data for this printer
    printer_flow = filter_flow_df[filter_flow_df['printer'] == printer_id].copy()
    printer_errors = error_df[error_df['printer'] == printer_id].copy()
    
    if printer_flow.empty:
        print(f"No filter flow data available for Printer {printer_id}")
        return
    
    # Create figure with 2x2 subplots
    fig, axes = plt.subplots(2, 2, figsize=(18, 14))
    fig.suptitle(f'Printer {printer_id} - Data Collection Analysis', fontsize=22)
    
    # 1. Data Timeline with Collection Density (Top Left)
    ax1 = axes[0, 0]
    
    # Get unique colors for this printer
    colors_in_printer = printer_flow['color'].unique()
    
    # Plot timeline showing when data was collected for each color
    for i, color in enumerate(colors_in_printer):
        color_data = printer_flow[printer_flow['color'] == color].sort_values('timestamp')
        
        if not color_data.empty:
            # Calculate data collection periods 
            color_data['next_timestamp'] = color_data['timestamp'].shift(-1)
            color_data['interval'] = (color_data['next_timestamp'] - color_data['timestamp']).dt.total_seconds() / 3600  # hours
            
            # Identify large gaps (e.g., more than 48 hours)
            gap_threshold = 48  # hours
            collection_periods = []
            current_period_start = color_data['timestamp'].iloc[0]
            
            for _, row in color_data.iterrows():
                if pd.notna(row['interval']) and row['interval'] > gap_threshold:
                    collection_periods.append((current_period_start, row['timestamp']))
                    current_period_start = row['next_timestamp']
            
            # Add the last period
            if len(color_data) > 0:
                collection_periods.append((current_period_start, color_data['timestamp'].iloc[-1]))
            
            # Plot collection periods
            for j, (start, end) in enumerate(collection_periods):
                ax1.plot([start, end], [i, i], linewidth=8, label=color if j == 0 else "")
            
            # Add color label - fixing the error by using date objects properly
            min_date = pd.Timestamp(ax1.get_xlim()[0], unit='D')
            ax1.text(min_date, i, color, va='center', ha='right', fontsize=10)
    
            # Plot collection periods
            for j, (start, end) in enumerate(collection_periods):
                # Use a consistent color for each ink type
                if color == 'C':
                    line_color = 'blue'
                elif color == 'M':
                    line_color = 'magenta'
                elif color == 'Y':
                    line_color = 'yellow'
                elif color == 'K':
                    line_color = 'black'
                elif color == 'CG':
                    line_color = 'cyan'
                else:
                    line_color = 'gray'
                    
                ax1.plot([start, end], [i, i], color=line_color, linewidth=8, 
                        label=color if j == 0 else "")
    
    ax1.set_title('Data Collection Periods by Color')
    ax1.set_xlabel('Date')
    ax1.set_yticks([])
    ax1.set_ylabel('Colors')
    
    # Format x-axis dates
    ax1.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
    ax1.xaxis.set_major_locator(mdates.MonthLocator(interval=3))
    plt.setp(ax1.xaxis.get_majorticklabels(), rotation=45, ha='right')
    
    # 2. Data Collection Interval Statistics (Top Right)
    ax2 = axes[0, 1]
    
    # Calculate interval statistics for each color
    interval_stats = []
    
    for color in colors_in_printer:
        color_data = printer_flow[printer_flow['color'] == color].sort_values('timestamp')
        
        if len(color_data) > 1:
            # Calculate time differences
            color_data['prev_timestamp'] = color_data['timestamp'].shift(1)
            color_data['interval'] = (color_data['timestamp'] - color_data['prev_timestamp']).dt.total_seconds() / 60  # in minutes
            
            # Filter out NaN and intervals larger than a week
            valid_intervals = color_data['interval'].dropna()
            valid_intervals = valid_intervals[valid_intervals < 10080]  # 7 days in minutes
            
            if len(valid_intervals) > 0:
                stats = {
                    'color': color,
                    'mean_interval_minutes': valid_intervals.mean(),
                    'median_interval_minutes': valid_intervals.median(),
                    'std_interval_minutes': valid_intervals.std(),
                    'min_interval_minutes': valid_intervals.min(),
                    'max_interval_minutes': valid_intervals.max(),
                    'data_points': len(color_data)
                }
                interval_stats.append(stats)
    
    if interval_stats:
        interval_df = pd.DataFrame(interval_stats)
        
        # Create barplot for mean intervals instead of boxplot
        sns.barplot(x='color', y='mean_interval_minutes', data=interval_df, ax=ax2)
        
        # Add data point counts
        for i, row in enumerate(interval_df.itertuples()):
            ax2.text(i, row.mean_interval_minutes + 1, f'n={row.data_points}', 
                     ha='center', va='bottom', fontsize=10)
        
        ax2.set_title('Mean Data Collection Interval by Color')
        ax2.set_ylabel('Average Interval (minutes)')
        ax2.set_xlabel('Ink Color')
    else:
        ax2.text(0.5, 0.5, 'Insufficient data for interval analysis', 
                 ha='center', va='center', fontsize=12)
    
    # 3. Interval Histogram (Bottom Left)
    ax3 = axes[1, 0]
    
    # Create histograms of intervals for each color
    colors_to_plot = min(len(colors_in_printer), 5)  # Limit to 5 colors to avoid overcrowding
    
    for color in list(colors_in_printer)[:colors_to_plot]:
        color_data = printer_flow[printer_flow['color'] == color].sort_values('timestamp')
        
        if len(color_data) > 1:
            # Calculate time differences
            color_data['prev_timestamp'] = color_data['timestamp'].shift(1)
            color_data['interval'] = (color_data['timestamp'] - color_data['prev_timestamp']).dt.total_seconds() / 60  # in minutes
            
            # Filter out NaN and intervals larger than a day
            valid_intervals = color_data['interval'].dropna()
            valid_intervals = valid_intervals[valid_intervals < 1440]  # 24 hours in minutes
            
            if len(valid_intervals) > 0:
                ax3.hist(valid_intervals, bins=30, alpha=0.5, label=color)
    
    ax3.set_title('Distribution of Data Collection Intervals')
    ax3.set_xlabel('Interval (minutes)')
    ax3.set_ylabel('Frequency')
    ax3.legend(title='Color')
    
    # 4. Monthly Data Volume (Bottom Right)
    ax4 = axes[1, 1]
    
    # Add month and year columns
    printer_flow['month_year'] = printer_flow['timestamp'].dt.to_period('M')
    
    try:
        # Count data points by month and color
        monthly_counts = printer_flow.groupby(['month_year', 'color']).size().reset_index(name='count')
        
        # Convert period to datetime for plotting
        monthly_counts['month_date'] = monthly_counts['month_year'].dt.to_timestamp()
        
        # Get unique months
        all_months = sorted(monthly_counts['month_date'].unique())
        
        # Plot monthly data volume
        if not monthly_counts.empty and len(all_months) > 1:
            for color in colors_in_printer:
                color_data = monthly_counts[monthly_counts['color'] == color]
                if not color_data.empty:
                    ax4.plot(color_data['month_date'], color_data['count'], 
                            marker='o', linewidth=2, markersize=6, label=color)
            
            ax4.set_title('Monthly Data Volume by Color')
            ax4.set_ylabel('Number of Data Points')
            ax4.set_xlabel('Month')
            ax4.legend(title='Color')
            
            # Format x-axis dates
            ax4.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
            ax4.xaxis.set_major_locator(mdates.MonthLocator(interval=2))
            plt.setp(ax4.xaxis.get_majorticklabels(), rotation=45, ha='right')
        else:
            ax4.text(0.5, 0.5, 'Insufficient data for monthly analysis', 
                    ha='center', va='center', fontsize=12)
    except Exception as e:
        ax4.text(0.5, 0.5, f'Error in monthly analysis: {str(e)}', 
                ha='center', va='center', fontsize=10)
    
    plt.tight_layout(rect=[0, 0, 1, 0.95])
    plt.show()
    
    # Print interval statistics as a table
    if interval_stats:
        print(f"\nPrinter {printer_id} Data Collection Interval Statistics:")
        interval_table = pd.DataFrame(interval_stats)
        interval_table = interval_table[['color', 'data_points', 'mean_interval_minutes', 
                                       'median_interval_minutes', 'min_interval_minutes', 
                                       'max_interval_minutes']]
        interval_table.columns = ['Color', 'Data Points', 'Mean Interval (min)', 
                                'Median Interval (min)', 'Min Interval (min)', 
                                'Max Interval (min)']
        return interval_table.round(2)
    return None

# Execute visualization for each printer
for printer_id in printers:
    print(f"\n{'='*50}\nAnalyzing Printer {printer_id}\n{'='*50}")
    try:
        stats_table = visualize_printer_data(printer_id)
        if stats_table is not None:
            display(stats_table)
    except Exception as e:
        print(f"Error analyzing printer {printer_id}: {str(e)}")